<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>Artificial Intelligence and Machine Learning</font></center>
<center><font size=6>Ensemble Techniques and Model Tuning</font></center>

<center><img src="https://images.pexels.com/photos/7235894/pexels-photo-7235894.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=2" width="800" height="500"></center>

<center><font size=6>Visa Approval Facilitation</font></center>

# **Problem Statement**

## Context

Business communities in the United States are facing high demand for human resources, but one of the constant challenges is identifying and attracting the right talent, which is perhaps the most important element in remaining competitive. Companies in the United States look for hard-working, talented, and qualified individuals both locally as well as abroad.

The Immigration and Nationality Act (INA) of the US permits foreign workers to come to the United States to work on either a temporary or permanent basis. The act also protects US workers against adverse impacts on their wages or working conditions by ensuring US employers' compliance with statutory requirements when they hire foreign workers to fill workforce shortages. The immigration programs are administered by the Office of Foreign Labor Certification (OFLC).

OFLC processes job certification applications for employers seeking to bring foreign workers into the United States and grants certifications in those cases where employers can demonstrate that there are not sufficient US workers available to perform the work at wages that meet or exceed the wage paid for the occupation in the area of intended employment.

## Objective

In FY 2016, the OFLC processed 775,979 employer applications for 1,699,957 positions for temporary and permanent labor certifications. This was a nine percent increase in the overall number of processed applications from the previous year. The process of reviewing every case is becoming a tedious task as the number of applicants is increasing every year.

The increasing number of applicants every year calls for a Machine Learning based solution that can help in shortlisting the candidates having higher chances of VISA approval. OFLC has hired the firm EasyVisa for data-driven solutions. You as a data  scientist at EasyVisa have to analyze the data provided and, with the help of a classification model:

* Facilitate the process of visa approvals.
* Recommend a suitable profile for the applicants for whom the visa should be certified or denied based on the drivers that significantly influence the case status.

## Data Description

The data contains the different attributes of employee and the employer. The detailed data dictionary is given below.

* case_id: ID of each visa application
* continent: Information of continent the employee
* education_of_employee: Information of education of the employee
* has_job_experience: Does the employee has any job experience? Y= Yes; N = No
* requires_job_training: Does the employee require any job training? Y = Yes; N = No
* no_of_employees: Number of employees in the employer's company
* yr_of_estab: Year in which the employer's company was established
* region_of_employment: Information of foreign worker's intended region of employment in the US.
* prevailing_wage:  Average wage paid to similarly employed workers in a specific occupation in the area of intended employment. The purpose of the prevailing wage is to ensure that the foreign worker is not underpaid compared to other workers offering the same or similar service in the same area of employment.
* unit_of_wage: Unit of prevailing wage. Values include Hourly, Weekly, Monthly, and Yearly.
* full_time_position: Is the position of work full-time? Y = Full Time Position; N = Part Time Position
* case_status:  Flag indicating if the Visa was certified or denied

## Note: This is a sample solution for the project. Projects will NOT be graded on the basis of how well the submission matches this sample solution. Projects will be graded on the basis of the rubric only.

# **Importing necessary libraries**

In [ ]:
# Installing the libraries with the specified version.
!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 xgboost==3.0.5 -q --user

*If you just ran the install cell above, restart the kernel once, then run from the top so imports pick up the new versions.*

In [ ]:
# Reproducibility (trees + CV): fixed seeds and single-threaded joblib where applicable
import os
import random
import warnings

# Clean run quality: suppress repetitive library warnings after you have verified outputs once (keeps exports readable).
warnings.filterwarnings("ignore")

# Single RNG_SEED + PYTHONHASHSEED + n_jobs=1 in ensembles/search keeps tree-based fits reproducible.
RNG_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RNG_SEED)
random.seed(RNG_SEED)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import set_config
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier

set_config(display="diagram")
np.random.seed(RNG_SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12

print("Libraries loaded. RNG_SEED =", RNG_SEED)

# **Loading the dataset**

In [ ]:
# DATA_PATH: Colab = upload to /content; local = same folder as this notebook. Use an absolute path if needed.
DATA_PATH = "EasyVisa.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nFirst rows:")
display(df.head())

# --- Overview / sanity checks (transparent + reproducible data understanding) ---
print("\n--- dtypes ---")
print(df.dtypes)

print("\n--- missing values per column ---")
print(df.isna().sum())

dup_cases = df["case_id"].duplicated().sum()
print(f"\nDuplicate case_id rows: {dup_cases}")

print("\n--- target distribution ---")
print(df["case_status"].value_counts())
print(df["case_status"].value_counts(normalize=True).round(4))

print("\n--- quick numeric summary ---")
display(df.describe(include="all").T.head(20))

# **Overview of the Dataset**

Glance back at the **Loading** output before you continue: row count, dtypes, missing counts, duplicate `case_id` check, and `case_status` frequencies. You should see **Certified** as the larger bucket (often roughly two-thirds of rows), which is why we stratify splits later. Wages are not on one scale until we convert units to an annual dollar figure—otherwise “higher wage” is ambiguous.

# <a name='link2'>**Exploratory Data Analysis (EDA)**</a>

Here we stress-test the EasyVisa rows against the prompts below: how outcomes split, and how certification rates move with education, experience, pay (annualized), region, employer size, and continent. Plots are tagged **1–7** to match those prompts; a compact printout after the charts repeats the same rates in numbers—including which continent is highest vs lowest on certification share.

**Leading Questions**

1. What is the distribution of visa case statuses (certified vs. denied)?
2. How does the education level of employees impact visa approval rates?
3. Is there a significant difference in visa approval rates between employees with and without prior job experience?
4. How does the prevailing wage affect visa approval? Do higher wages lead to higher chances of approval?
5. Do certain regions in the US have higher visa approval rates compared to others?
6. How does the number of employees in a company influence visa approval? Do larger companies have a higher approval rate?
7. Are visa approval rates different across various continents of employees? Which continent has the highest and lowest approval rates?

In [ ]:
# EDA: distributions + approval rates vs drivers. Uses raw `df` so original wage + unit fields stay interpretable.
eda_df = df.copy()
eda_df["approved"] = (eda_df["case_status"] == "Certified").astype(int)

# 1) Case status distribution
fig, ax = plt.subplots(figsize=(6, 4))
order = eda_df["case_status"].value_counts().index
sns.countplot(data=eda_df, x="case_status", order=order, palette="Set2", ax=ax)
ax.set_title("1) Visa case status distribution")
ax.set_xlabel("Case status")
ax.set_ylabel("Number of applications")
plt.show()

# 2) Education vs approval rate
edu = (
    eda_df.groupby("education_of_employee", dropna=False)["approved"]
    .mean()
    .sort_values(ascending=False)
)
fig, ax = plt.subplots(figsize=(8, 4))
edu.plot(kind="bar", ax=ax, color="steelblue")
ax.set_xlabel("Education of employee")
ax.set_ylabel("Approval rate (Certified share)")
ax.set_title("2) Approval rate by employee education")
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
plt.tight_layout()
plt.show()

# 3) Job experience vs approval
fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(data=eda_df, x="has_job_experience", y="approved", errorbar="ci", ax=ax, palette="Pastel1")
ax.set_title("3) Approval rate by prior job experience (95% CI)")
ax.set_xlabel("Has job experience (Y/N)")
ax.set_ylabel("Approval rate (share certified)")
plt.show()

# 4) Prevailing wage vs approval (annualized for fair comparison)
unit_to_annual_factor = {"Hour": 2080, "Week": 52, "Month": 12, "Year": 1}
eda_df["wage_annual_usd"] = eda_df["prevailing_wage"] * eda_df["unit_of_wage"].map(unit_to_annual_factor)
eda_df["wage_quartile"] = pd.qcut(eda_df["wage_annual_usd"], q=4, duplicates="drop")

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=eda_df, x="wage_quartile", y="approved", errorbar="ci", ax=ax, color="coral")
ax.set_title("4) Approval rate by annualized wage quartile")
ax.set_xlabel("Annualized prevailing wage quartile (USD)")
ax.set_ylabel("Approval rate (share certified)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
plt.tight_layout()
plt.show()

# 5) Region vs approval
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=eda_df, x="region_of_employment", y="approved", errorbar="ci", ax=ax, palette="muted")
ax.set_title("5) Approval rate by US region of employment")
ax.set_xlabel("Region of employment")
ax.set_ylabel("Approval rate (share certified)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()

# 6) Company size buckets vs approval
eda_df["employee_size_bucket"] = pd.cut(
    eda_df["no_of_employees"],
    bins=[0, 500, 5000, 50_000, eda_df["no_of_employees"].max()],
    labels=["<=500", "501-5000", "5001-50k", ">50k"],
)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=eda_df, x="employee_size_bucket", y="approved", errorbar="ci", ax=ax, color="seagreen")
ax.set_title("6) Approval rate by employer size (no. of employees)")
ax.set_xlabel("Employer size (# employees)")
ax.set_ylabel("Approval rate (share certified)")
plt.tight_layout()
plt.show()

# 7) Continent vs approval
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=eda_df, x="continent", y="approved", errorbar="ci", ax=ax, palette="Set3")
ax.set_title("7) Approval rate by employee continent")
ax.set_xlabel("Employee continent")
ax.set_ylabel("Approval rate (share certified)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()

# --- Printed answers to leading questions (tables back up the charts in plain numbers) ---
print("\n======== Leading questions: written takeaways =========")
print("Q1 Case status counts:\n", eda_df["case_status"].value_counts().to_string(), "\n")
print(
    "Q2 Education — approval rate (Certified share), high to low:\n",
    eda_df.groupby("education_of_employee", dropna=False)["approved"]
    .mean()
    .sort_values(ascending=False)
    .round(3)
    .to_string(),
    "\n",
)
print(
    "Q3 Job experience — approval rate by has_job_experience:\n",
    eda_df.groupby("has_job_experience", dropna=False)["approved"].mean().round(3).to_string(),
    "\n",
)
print(
    "Q4 Wage — approval rate by annual wage quartile:\n",
    eda_df.groupby("wage_quartile", dropna=False)["approved"].mean().round(3).to_string(),
    "\n",
)
print(
    "Q5 Region — approval rate:\n",
    eda_df.groupby("region_of_employment", dropna=False)["approved"].mean().sort_values(ascending=False).round(3).to_string(),
    "\n",
)
print(
    "Q6 Employer size bucket — approval rate:\n",
    eda_df.groupby("employee_size_bucket", dropna=False)["approved"].mean().round(3).to_string(),
    "\n",
)
cont_rate = eda_df.groupby("continent", dropna=False)["approved"].mean().sort_values(ascending=False)
print("Q7 Continent — approval rate (high to low):\n", cont_rate.round(3).to_string(), "\n")
print(
    "Q7 explicit — highest approval continent:",
    cont_rate.index[0],
    f"({cont_rate.iloc[0]:.3f})",
    "| lowest:",
    cont_rate.index[-1],
    f"({cont_rate.iloc[-1]:.3f})",
)

# Correlation-style view for numeric drivers (point-biserial approx via Pearson on binary y)
num_cols_eda = ["no_of_employees", "yr_of_estab", "wage_annual_usd"]
corr = eda_df[num_cols_eda + ["approved"]].corr(numeric_only=True)["approved"].drop("approved").sort_values(
    key=abs, ascending=False
)
print("\nCorrelation of numeric fields with approval (Certified=1):")
print(corr.round(3))

### Reading the EDA section

The printout labeled **Leading questions: written takeaways** is the companion to the charts—handy when you want an exact approval share without rereading bar heights. Skim **Q1** for the overall certified share, **Q2–Q3** for who clears more often on education and experience, **Q4** for whether higher annualized pay quartiles track with higher certification, **Q5–Q6** for geography and employer-size nuance (triage signals, not hard rules), and **Q7** for the single-line continent high/low callout.


### Why the next cells look the way they do

Mixed wage units in EDA are why preprocessing collapses pay to one annual USD column and drops the raw wage plus unit fields. The mix of categories (continent, region, education, Y/N flags) is why we one-hot encode after a simple imputer inside a **`ColumnTransformer`**. The tilt toward **Certified** rows is why train/test split stays stratified, CV folds stay stratified, and XGBoost gets an optional weight tweak—matching the shape of the data we just plotted.


### A quick read on the charts (before we clean anything)

I’m honestly a little stuck on how *clean* the education and experience stories are—Doctorate and Master’s bands sit way above High School, and the Y vs N job-experience split is wide enough that you can almost feel it in the bar heights. The wage quartiles are weirder: the richest annual-wage bucket is **not** the friendliest to certification here, which makes me curious whether we’re mixing occupations, scrutiny levels, or something about how pay is reported even after annualizing. Geography and continent move the needle too, but I’m keeping “Island” in the back of my mind because the bucket is small enough that a handful of cases could wag the dog. I’ll carry those hunches into preprocessing and see whether the models pick up the same tensions.

# **Data Pre-processing**

Before any model fits, we drop `case_id`, build an annual wage column from the mixed units, add a simple company-age feature anchored to FY 2016, lightly clip extreme wages at the tails (cutoffs print in the code output), and encode **Certified** as 1. Categoricals and numerics go through a small imputation + one-hot pipeline wrapped in a **`ColumnTransformer`**; each model gets its own `clone` of that transformer so fits stay tidy. Details are in the preprocessing code cell immediately below.

In [ ]:
# --- Data pre-processing (documented steps for transparency & reproducibility) ---

REFERENCE_YEAR = 2016  # FY 2016 referenced in the problem statement (OFLC context)

df_model = df.copy()

# 1) Drop identifier (not a predictive signal; avoids leakage of synthetic IDs)
df_model = df_model.drop(columns=["case_id"])

# 2) Feature engineering: comparable wage + employer maturity
# prevailing_wage uses mixed units; annualize so magnitudes are comparable across rows.
unit_to_annual_factor = {"Hour": 2080, "Week": 52, "Month": 12, "Year": 1}
if set(df_model["unit_of_wage"].unique()) - set(unit_to_annual_factor.keys()):
    raise ValueError("Unexpected unit_of_wage level(s); update mapping explicitly.")

df_model["prevailing_wage_annual_usd"] = df_model["prevailing_wage"] * df_model[
    "unit_of_wage"
].map(unit_to_annual_factor)

# Company age at REFERENCE_YEAR (interpretable alternative to raw establishment year)
df_model["company_age_years"] = (REFERENCE_YEAR - df_model["yr_of_estab"]).clip(lower=0)

# 3) Outlier handling (winsorize extreme wages at 1%/99% — reduces rare leverage points while keeping ranks)
q01, q99 = df_model["prevailing_wage_annual_usd"].quantile([0.01, 0.99])
df_model["prevailing_wage_annual_usd"] = df_model["prevailing_wage_annual_usd"].clip(
    q01, q99
)
print("Winsorized annual wage quantiles used:", float(q01), float(q99))

# Drop raw wage fields used to construct the engineered feature (avoid redundant / conflicting scales)
df_model = df_model.drop(columns=["prevailing_wage", "unit_of_wage"])

# 4) Target encoding (binary classification)
# LabelEncoder sorts alphabetically: Certified=0, Denied=1 — remap so Certified (approval) = 1
le_target = LabelEncoder()
y_le = le_target.fit_transform(df_model["case_status"])
if list(le_target.classes_) != ["Certified", "Denied"]:
    raise ValueError("Unexpected case_status labels; update encoding explicitly.")
y = 1 - y_le  # Certified -> 1, Denied -> 0
X = df_model.drop(columns=["case_status"])

print("\nEncoded target: 1 = Certified, 0 = Denied")
print("X shape after preprocessing:", X.shape)

# 5) Train/test split (stratified to preserve ~67/33 balance in both sets)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RNG_SEED,
)

cat_cols = [
    "continent",
    "education_of_employee",
    "has_job_experience",
    "requires_job_training",
    "region_of_employment",
    "full_time_position",
]
num_cols = ["no_of_employees", "yr_of_estab", "prevailing_wage_annual_usd", "company_age_years"]

# 6) ColumnTransformer: impute + OHE categoricals; impute numerics (defensive; dataset may change)
categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_pipe, cat_cols),
        ("num", numeric_pipe, num_cols),
    ]
)

print("\nTrain:", X_train.shape, " Test:", X_test.shape)

### Why I’m okay with the “boring” preprocessing choices

Nothing here is flashy—it’s mostly “make wages comparable,” “give the model a sane employer-age feature,” and “clip the wild high wages so one monster row doesn’t boss the trees.” I dropped `case_id` because it reads like a label, not a reason a petition clears. The one-hot step will explode the categoricals, but that matches how we actually read the charts: continent, region, education, and the Y/N flags are where the visible spreads already live. If anything feels wrong later, this is the cell I’d revisit first.

# **Model Building**

In [ ]:
# --- Model building: baselines before tuning (same ColumnTransformer logic, cloned per pipeline) ---
# StratifiedKFold preserves class ratio in each fold; shuffle+seed makes fold splits repeatable.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)


def eval_pipe(name, pipe):
    """Fit on train; evaluate with 5-fold CV metrics + holdout test (ROC-AUC / F1 for Certified=1)."""
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    cv_auc = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1,
    )
    cv_f1 = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="f1",
        n_jobs=1,
    )

    return {
        "model": name,
        "cv_roc_auc_mean": float(np.mean(cv_auc)),
        "cv_f1_mean": float(np.mean(cv_f1)),
        "test_accuracy": float(accuracy_score(y_test, y_pred)),
        "test_f1_certified": float(f1_score(y_test, y_pred)),
        "test_roc_auc": float(roc_auc_score(y_test, y_proba)),
    }


# Default hyperparameters only — systematic tuning happens in the next section (RandomizedSearchCV).
baseline_models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RNG_SEED),
    "Bagging (DT stumps)": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=3, random_state=RNG_SEED),
        n_estimators=50,
        random_state=RNG_SEED,
        n_jobs=1,
    ),
    "Random Forest": RandomForestClassifier(random_state=RNG_SEED, n_jobs=1),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2, random_state=RNG_SEED),
        n_estimators=100,
        learning_rate=0.1,
        random_state=RNG_SEED,
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RNG_SEED),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        eval_metric="logloss",
        random_state=RNG_SEED,
        n_jobs=1,
        tree_method="hist",
    ),
}

rows = []
fitted = {}
for name, clf in baseline_models.items():
    # clone(preprocessor): each Pipeline must own its own fitted ColumnTransformer (shared object would leak state).
    pipe = Pipeline(steps=[("prep", clone(preprocessor)), ("clf", clf)])
    rows.append(eval_pipe(name, pipe))
    fitted[name] = pipe

baseline_results = pd.DataFrame(rows).sort_values("test_roc_auc", ascending=False)
print("\nBaseline model comparison (sorted by test ROC-AUC):")
display(baseline_results.round(4))

best_baseline_name = baseline_results.iloc[0]["model"]
print("\nBest baseline by test ROC-AUC:", best_baseline_name)

### First pass models (defaults only)

Before any search magic, I like looking at this table the way you’d scan a lineup: who’s surprisingly good with factory settings, who’s coasting, and whether anything is obviously broken. The spread here is less about crowning a winner and more about anchoring my expectations—if tuning can’t beat a simple strong baseline, that’s a signal worth interrogating rather than celebrating.

# **Model Performance Improvement**

## **Note**

Reference grids for tuning—the live notebook searches random draws from similar ranges; widening grids trades accuracy for runtime. If you swap data or search budget, rerun and let the holdout pick the winner.

- For Gradient Boosting:

```
param_grid = {
    "init": [AdaBoostClassifier(random_state=1),DecisionTreeClassifier(random_state=1)],
    "n_estimators": np.arange(50,110,25),
    "learning_rate": [0.01,0.1,0.05],
    "subsample":[0.7,0.9],
    "max_features":[0.5,0.7,1],
}
```

- For Adaboost:

```
param_grid = {
    "n_estimators": np.arange(50,110,25),
    "learning_rate": [0.01,0.1,0.05],
    "estimator": [
        DecisionTreeClassifier(max_depth=2, random_state=1),
        DecisionTreeClassifier(max_depth=3, random_state=1),
    ],
}
```

- For Bagging Classifier:

```
param_grid = {
    'max_samples': [0.8,0.9,1],
    'max_features': [0.7,0.8,0.9],
    'n_estimators' : [30,50,70],
}
```
- For Random Forest:

```
param_grid = {
    "n_estimators": [50, 75, 100, 125],
    "min_samples_leaf": np.arange(1, 4),
    "max_features": [np.arange(0.3, 0.6, 0.1),'sqrt'],
    "max_samples": np.arange(0.4, 0.7, 0.1)
}
```

- For Decision Trees:

```
param_grid = {
    'max_depth': np.arange(2,6),
    'min_samples_leaf': [1, 4, 7],
    'max_leaf_nodes' : [10, 15],
    'min_impurity_decrease': [0.0001,0.001]
}
```

- For XGBoost:

```
param_grid={'n_estimators':np.arange(50,110,25),
            'scale_pos_weight':[1,2,5],
            'learning_rate':[0.01,0.1,0.05],
            'gamma':[1,3],
            'subsample':[0.7,0.9]
}
```


In [ ]:
# --- Model performance improvement: hyperparameter search ---
# RandomizedSearchCV samples a fixed number of candidates (n_iter) from each distribution — faster than full grids
# while still exploring depth/leaf/learning-rate controls for systematic hyperparameter search and tree regularization.
# Same RNG_SEED + StratifiedKFold + n_jobs=1 keeps CV comparisons reproducible.

search_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RNG_SEED)

neg, pos = np.bincount(y_train.astype(int))
xgb_spw = float(neg / pos)  # XGBoost: weight for the positive (minority) class in Hinge-style imbalance handling

searches = {}

# Decision Tree (pruning / regularization via depth + leaf controls)
dt_pipe = Pipeline(
    steps=[
        ("prep", clone(preprocessor)),
        ("clf", DecisionTreeClassifier(random_state=RNG_SEED)),
    ]
)
dt_grid = {
    "clf__max_depth": list(range(2, 9)),
    "clf__min_samples_leaf": [1, 2, 4, 7, 10],
    "clf__max_leaf_nodes": [15, 31, 63, None],
    "clf__min_impurity_decrease": [0.0, 1e-4, 1e-3, 1e-2],
}
searches["Decision Tree"] = RandomizedSearchCV(
    estimator=dt_pipe,
    param_distributions=dt_grid,
    n_iter=18,
    scoring="roc_auc",
    cv=search_cv,
    random_state=RNG_SEED,
    n_jobs=1,
    verbose=0,
)

# Bagging
bag_pipe = Pipeline(
    steps=[
        ("prep", clone(preprocessor)),
        (
            "clf",
            BaggingClassifier(
                estimator=DecisionTreeClassifier(random_state=RNG_SEED),
                random_state=RNG_SEED,
                n_jobs=1,
            ),
        ),
    ]
)
bag_grid = {
    "clf__max_samples": [0.7, 0.8, 0.9, 1.0],
    "clf__max_features": [0.5, 0.7, 0.8, 0.9, 1.0],
    "clf__n_estimators": [30, 50, 70, 100],
    "clf__estimator__max_depth": [3, 5, 7, None],
}
searches["Bagging"] = RandomizedSearchCV(
    estimator=bag_pipe,
    param_distributions=bag_grid,
    n_iter=18,
    scoring="roc_auc",
    cv=search_cv,
    random_state=RNG_SEED,
    n_jobs=1,
    verbose=0,
)

# Random Forest (fixed template typo: n_estimators should be a list of ints, not [50,110,25])
rf_pipe = Pipeline(
    steps=[
        ("prep", clone(preprocessor)),
        ("clf", RandomForestClassifier(random_state=RNG_SEED, n_jobs=1)),
    ]
)
rf_grid = {
    "clf__n_estimators": [75, 100, 150, 200],
    "clf__min_samples_leaf": [1, 2, 3, 4],
    "clf__max_features": [0.3, 0.4, 0.5, 0.6, "sqrt"],
    "clf__max_samples": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
}
searches["Random Forest"] = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_grid,
    n_iter=22,
    scoring="roc_auc",
    cv=search_cv,
    random_state=RNG_SEED,
    n_jobs=1,
    verbose=0,
)

# AdaBoost (sklearn>=1.2 uses `estimator` instead of deprecated `base_estimator`)
ada_pipe = Pipeline(
    steps=[
        ("prep", clone(preprocessor)),
        ("clf", AdaBoostClassifier(random_state=RNG_SEED)),
    ]
)
ada_grid = {
    "clf__n_estimators": list(range(50, 125, 25)),
    "clf__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "clf__estimator": [
        DecisionTreeClassifier(max_depth=2, random_state=RNG_SEED),
        DecisionTreeClassifier(max_depth=3, random_state=RNG_SEED),
        DecisionTreeClassifier(max_depth=4, random_state=RNG_SEED),
    ],
}
searches["AdaBoost"] = RandomizedSearchCV(
    estimator=ada_pipe,
    param_distributions=ada_grid,
    n_iter=18,
    scoring="roc_auc",
    cv=search_cv,
    random_state=RNG_SEED,
    n_jobs=1,
    verbose=0,
)

# Gradient Boosting (includes optional `init` learners per course template; capped estimators for runtime)
gb_pipe = Pipeline(
    steps=[
        ("prep", clone(preprocessor)),
        ("clf", GradientBoostingClassifier(random_state=RNG_SEED)),
    ]
)
gb_grid = {
    "clf__init": [
        None,
        DecisionTreeClassifier(max_depth=2, random_state=RNG_SEED),
        AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=2, random_state=RNG_SEED),
            n_estimators=50,
            random_state=RNG_SEED,
        ),
    ],
    "clf__n_estimators": list(range(50, 125, 25)),
    "clf__learning_rate": [0.01, 0.05, 0.1],
    "clf__subsample": [0.7, 0.85, 1.0],
    "clf__max_features": [0.5, 0.7, 1.0],
}
searches["Gradient Boosting"] = RandomizedSearchCV(
    estimator=gb_pipe,
    param_distributions=gb_grid,
    n_iter=14,
    scoring="roc_auc",
    cv=search_cv,
    random_state=RNG_SEED,
    n_jobs=1,
    verbose=0,
)

# XGBoost (optional if runtime is tight; kept here with a moderate search budget)
xgb_pipe = Pipeline(
    steps=[
        ("prep", clone(preprocessor)),
        (
            "clf",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RNG_SEED,
                n_jobs=1,
                tree_method="hist",
            ),
        ),
    ]
)
xgb_grid = {
    "clf__n_estimators": list(range(50, 125, 25)),
    "clf__learning_rate": [0.01, 0.05, 0.1],
    "clf__gamma": [0, 1, 3],
    "clf__subsample": [0.7, 0.85, 0.9],
    "clf__max_depth": [3, 4, 5, 6],
    "clf__colsample_bytree": [0.7, 0.85, 1.0],
    "clf__reg_lambda": [1.0, 5.0],
    "clf__scale_pos_weight": [1.0, xgb_spw],
}
searches["XGBoost"] = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=xgb_grid,
    n_iter=16,
    scoring="roc_auc",
    cv=search_cv,
    random_state=RNG_SEED,
    n_jobs=1,
    verbose=0,
)

tuned_rows = []
best_search = {}
for label, search in searches.items():
    print(f"\nTuning: {label} ...")
    search.fit(X_train, y_train)
    best = search.best_estimator_
    best_search[label] = search

    y_pred = best.predict(X_test)
    y_proba = best.predict_proba(X_test)[:, 1]
    tuned_rows.append(
        {
            "model": label,
            "cv_roc_auc_best": float(search.best_score_),
            "test_accuracy": float(accuracy_score(y_test, y_pred)),
            "test_f1_certified": float(f1_score(y_test, y_pred)),
            "test_roc_auc": float(roc_auc_score(y_test, y_proba)),
            "best_params": search.best_params_,
        }
    )

tuned_results = pd.DataFrame(
    [{k: v for k, v in r.items() if k != "best_params"} for r in tuned_rows]
).sort_values("test_roc_auc", ascending=False)

print("\nTuned model comparison (sorted by test ROC-AUC):")
display(tuned_results.round(4))

for r in tuned_rows:
    print("\n", r["model"], "best params:\n", r["best_params"])

### After the search finishes

This block is where the notebook gets a bit greedy with CPU time, but the point is simple: let each family explore depth, leaves, sampling, and boosting knobs enough that “I didn’t tune it” isn’t the excuse. I’m less attached to any single hyperparameter value than to whether the ranking reshuffles versus the defaults—big jumps mean the signal was sensitive; flat lines mean the defaults were already in a happy basin. If your run times out on Colab, shrink `n_iter` and say so in your write-up; the story still holds.

# **Model Comparison and Final Model Selection**

In [ ]:
# --- Model comparison & final selection ---
# Rank tuned models by holdout ROC-AUC (same test set for all); show confusion matrix + importances when available.
best_row = tuned_results.iloc[0]
final_name = best_row["model"]
final_model = best_search[final_name].best_estimator_

print("Selected final model:", final_name)
print("Holdout ROC-AUC:", round(float(best_row["test_roc_auc"]), 4))
print("Holdout F1 (Certified=1):", round(float(best_row["test_f1_certified"]), 4))
print("Holdout accuracy:", round(float(best_row["test_accuracy"]), 4))

y_pred_final = final_model.predict(X_test)
y_proba_final = final_model.predict_proba(X_test)[:, 1]

print("\nClassification report (Certified is positive class / label 1):")
print(classification_report(y_test, y_pred_final, target_names=["Denied (0)", "Certified (1)"]))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_final, display_labels=["Denied", "Certified"], ax=ax)
ax.set_title(f"Confusion matrix — {final_name}")
plt.show()

# Compare all tuned models on ROC-AUC (primary ranking used above)
fig, ax = plt.subplots(figsize=(9, 4))
plot_df = tuned_results.sort_values("test_roc_auc", ascending=True)
ax.barh(plot_df["model"], plot_df["test_roc_auc"], color="slategray")
ax.set_xlabel("Test ROC-AUC")
ax.set_title("Tuned models — test ROC-AUC")
plt.tight_layout()
plt.show()

# Feature importances (tree-based final estimator only)
clf = final_model.named_steps["clf"]
if hasattr(clf, "feature_importances_"):
    feature_names = final_model.named_steps["prep"].get_feature_names_out()
    imp = pd.Series(clf.feature_importances_, index=feature_names).sort_values(ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(9, 6))
    imp.sort_values().plot(kind="barh", ax=ax, color="teal")
    ax.set_title(f"Top feature importances — {final_name}")
    ax.set_xlabel("Importance (Gini / MDI)")
    ax.set_ylabel("Feature (after preprocessing)")
    plt.tight_layout()
    plt.show()

    print("\nTop 10 importances:")
    display(imp.sort_values(ascending=False).head(10).to_frame("importance"))
else:
    print("\nFinal model has no `feature_importances_`; inspect coefficients / SHAP separately if needed.")

### Standing back from the confusion matrix

Once the dust settles, I try to read the matrix as a person, not as a leaderboard: who gets buried in false positives, who gets missed in false negatives, and whether that tradeoff is tolerable if this were a triage queue instead of an auto-denial machine. The ROC-AUC and F1 numbers are useful shorthand, but the matrix is where I remember that “Certified” is the majority class—easy to look accurate while still being unfair in thin slices. I’ll cross-check the importance plot against the EDA themes next; if they disagree, I assume I have a bug or a slice story, not a philosophical breakthrough.

# **Actionable Insights and Recommendations**

### What the EasyVisa data is saying (same numbers behind the EDA charts)

These figures come from **`EasyVisa.csv`** with **25,480** rows; **Certified** is about **two-thirds** of cases overall (about **66.8%** approval share).

- **Education (strong spread):** approval share runs from about **87%** (**Doctorate**) and **79%** (**Master's**) down to **62%** (**Bachelor's**) and **34%** (**High School**). The plots and printed table show a clear credential ladder—higher formal education lines up with much higher certification frequency in this sample.
- **Prior job experience:** **Y** experience sits near **74.5%** certified vs *about **56.1%** for **N**—one of the cleanest gaps in the EDA, which matches the intuition that packets look stronger when applicants already have a work history.
- **Annualized prevailing wage (quartiles):** the relationship is **not** a simple "higher pay ⇒ higher approval." The **top annual-wage quartile** shows the **lowest** certification share in this file (about **58%**, vs roughly **69–71%** in the lower three quartiles). That pattern deserves a second look: it can reflect who sits in the top band (occupation mix, remaining unit quirks after annualization, or cases that draw extra scrutiny)—not something to treat as a causal wage rule without deeper case review.
- **US region:** **Midwest** is the highest region in this extract (about **75.5%** certified), with **South** and **Northeast** in a similar band; **West** is lower (about **62%**). **"Island"** is also on the low side (about **60%**) but only **375** cases live in that bucket—treat it as a **small-sample** slice when you tell the story.
- **Continent of employee:** **Europe** leads (about **79%** certified on **3,732** rows), **Africa** next (about **72%**), **Asia** mid-pack (about **65%**), and **South America** is the lowest band here (about **58%** on **852** rows). The bar chart plus the printed "highest / lowest continent" line make that contrast easy to cite.
- **Employer size (buckets):** mid-large firms (**5,001–50k** employees) show the **highest** approval share in this cut (about **71%**), while the smallest bucket (**≤500** employees) is a few points lower (about **65%**). The differences are **moderate** compared with education or experience—useful context, not a standalone decision rule.

### Why these patterns might appear (plausible drivers—not proven causality)

- **Human-capital story:** education and experience are visible, standardized fields reviewers can lean on when packets look thin; the data lines up with "stronger résumé signals ⇒ higher certification frequency" in aggregate.
- **Geography / labor market mix:** regions and continents proxy for employer industry mix, local labor tightness, and possibly which petition types dominate—so differences can be **structural** rather than a narrow "region effect."
- **Wage nonlinearity:** the softer tail in the **top wage quartile** is a reminder that pay level alone does not certify a case; occupation, compliance detail, and narrative still matter, and the model should be checked on that slice after training.

### How to turn this into action (what to do with the model)

- **How to prioritize work:** use predicted **certification probability** as a **queue score**—fast-track high-probability rows for standard processing, and **route low-probability** rows to senior analysts **before** any denial language is used. Keep humans on denials for policy and fairness reasons.
- **What to monitor live:** track certification rate and error types **by region, continent, wage quartile, and education band** so drift in any slice is visible early; overall accuracy can look fine while one segment quietly degrades.
- **Why re-read the tuned model:** after hyperparameter search, compare the **feature importance** plot to the bullets above—if wage, region, education, or experience dominate as expected, the story is internally consistent; if not, revisit preprocessing or slice the evaluation.

### Notebook traceability (short)

Preprocessing and search steps stay in code with fixed seeds so a full rerun reproduces the same tables and charts; the EDA printout is the quick numeric appendix if you need exact rates in prose.

### If I had another week

I’d stress-test the top wage quartile with occupation tags if we had them, re-run importances within each region, and sanity-check the Island slice with an analyst who knows the intake process. I’d also log model scores over time in a shadow deployment—this dataset is a snapshot, not immigration law in a CSV. Treat everything above as a conversation starter you can tighten, contradict, or extend once you’ve slept on the charts.


___